# 3. Dimensionality reduction: CNN2d latent landscape (6D) <a id="3"></a>
Experiment / variant: train a single `CNN2d_AE` with a **6-dimensional** latent space on the same fitted CG activation loops used by `07-PCAClusteringVsKinCore` and `07b-AutoencoderBenchmark`, reproduce the full reconstruction-metric suite, and map the latent space with one RMSD landscape per dimension pair.

Where `07b` sweeps models and latent sizes, this notebook drills into one model: with six latent dimensions there is no single scatter plot to look at, so the latent space is shown as the `C(6, 2) = 15` pairwise projections, each a 2-D slice with the other four dimensions pinned to the median of the encoded training set.

## Table of contents

- [0. Paths and config](#0-paths-and-config)
- [1. Train the 6D CNN2d autoencoder](#1-train-the-6d-cnn2d-autoencoder)
- [2. Training history](#2-training-history)
- [3. Reconstruction metrics](#3-reconstruction-metrics)
  - [3.1 Per-frame RMSD](#31-per-frame-rmsd)
  - [3.2 Radius of gyration](#32-radius-of-gyration)
  - [3.3 Per-residue RMSF](#33-per-residue-rmsf)
  - [3.4 Cα bond lengths](#34-ca-bond-lengths)
  - [3.5 Cα angles](#35-ca-angles)
  - [3.6 Cα pseudo-dihedrals](#36-ca-pseudo-dihedrals)
  - [3.7 Writhe chirality](#37-writhe-chirality)
- [4. The 6D latent space](#4-the-6d-latent-space)
  - [4.1 Encode the train and validation splits](#41-encode-the-train-and-validation-splits)
  - [4.2 Reconstruction-error violins](#42-reconstruction-error-violins)
  - [4.3 Pairwise RMSD landscapes](#43-pairwise-rmsd-landscapes)
  - [4.4 Close-up on one dimension pair](#44-close-up-on-one-dimension-pair)
- [5. Summary](#5-summary)

## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/04c-CNN2dLatentLandscape.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["04c-CNN2dLatentLandscape.ipynb"]
  m_ae["replacingPCA/autoencoder_workflow"]
  m_land["replacingPCA/latent_landscape"]
  m_export["replacingPCA/ae_aligned_export"]
  m_chir["replacingPCA/ae_writhe_chirality"]
  m_small["replacingPCA/small_foldingnet_latent"]
  m_wrch2["replacingPCA/wrCNN2D_ch2"]
  m_wr["replacingPCA/wrCNN2D"]
  m_trainer["replacingPCA/wrTrainer"]
  m_util["utilities"]
  m_wr --> m_wrch2
  m_wr --> m_trainer
  m_wr --> m_chir
  m_export --> m_chir
  m_wrch2 --> m_ae
  m_trainer --> m_ae
  m_small --> m_ae
  m_export --> m_ae
  m_chir --> m_ae
  m_land --> m_ae
  m_util --> m_ae
  m_ae --> NB
```
-->

## 0. Paths and config <a id="0-paths-and-config"></a>

Point at the August `05a` fitted CG loops and a local `Results/cnn2d_latent6/` tree for checkpoints, aligned PDBs, latent CSVs, and plots. Run with cwd = repository root (`workflowAugust2026/`), kernel = `molearn_latest`. CUDA is required.

`Results/activation_segments/fitted/` is the fixed-size product of `04a` alignment and `05a` coarse-graining: every structure has the same number of Cα atoms. That matters here, because `CNN2d_AE` encodes an `n × n` distance-matrix image and so needs a single `dm_dim` for the whole dataset. The variable-length chains in `Results/Bounds_CAfilter_chains/` cannot be fed to this model directly.

In [ ]:
import os

from workflow.replacingPCA.autoencoder_workflow import AutoencoderWorkflow

DATA_DIR = os.path.abspath("Results/activation_segments/fitted/")
OUT_DIR = os.path.abspath("Results/cnn2d_latent6/")
SUBFOLDER = "cnn2d_latent6"
ATOM_SELECTION = ["CA"]

LATENT_DIM = 6
MAX_EPOCHS = 32
PATIENCE = 32
MANUAL_SEED = 25
BATCH_SIZE = 8
VALIDATION_SPLIT = 0.1

# Landscape grid: 30 x 30 per pair x 15 pairs = 13,500 decode-encode-decode
# round trips, which is a few seconds on the GPU.
GRID_SIZE = 30
GRID_PADDING = 0.1

assert os.path.isdir(DATA_DIR), f"Missing data directory: {DATA_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)
n_pdbs = len(
    [f for f in os.listdir(DATA_DIR) if f.endswith(".pdb") and f != "combined.pdb"]
)
print(f"DATA_DIR   = {DATA_DIR}")
print(f"OUT_DIR    = {OUT_DIR}")
print(f"n_pdbs     = {n_pdbs}")
print(f"LATENT_DIM = {LATENT_DIM}")

## 1. Train the 6D CNN2d autoencoder <a id="1-train-the-6d-cnn2d-autoencoder"></a>

Same single-run flow as the `07b` baselines, with `latent_dim=6` instead of 2:

1. **prepare_data** — concatenate the per-structure PDBs into `combined.pdb`, select Cα, standardize to `(x − mean) / std`.
2. **train_cnn2d_ae** — distance matrix → latent → coordinates, molearn `Trainer` with an MSE loss. The workflow restarts the epoch loop while the best validation loss keeps improving, so the effective epoch count is a multiple of `MAX_EPOCHS`.
3. **setup_analysis** — rebuild the exact train/validation split used during training (same `manual_seed`), so the latent scatter below is split-faithful.

In [ ]:
# device=None picks the CUDA GPU with the most free memory.
wf = AutoencoderWorkflow(
    folder_name=DATA_DIR,
    output_base_dir=OUT_DIR,
    manual_seed=MANUAL_SEED,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
)
wf.prepare_data(atom_selection=ATOM_SELECTION)
wf.train_cnn2d_ae(
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    latent_dim=LATENT_DIM,
    init_c=32,
    m=2,
    min_size=9,
    output_subfolder=SUBFOLDER,
)

In [ ]:
wf.setup_analysis(atom_selection=ATOM_SELECTION)
print(f"train frames: {tuple(wf.data_train.shape)}")
print(f"valid frames: {tuple(wf.data_valid.shape)}")

## 2. Training history <a id="2-training-history"></a>

Train and validation loss per epoch from the molearn log. Only the latest epoch-loop segment is plotted, so the restart behaviour above does not stack multiple curves on top of each other.

In [ ]:
wf.plot_training_history(log_filename="log.dat", show=True);

## 3. Reconstruction metrics <a id="3-reconstruction-metrics"></a>

`export_kabsch_aligned_datasets` decodes both splits, Kabsch-aligns each decoded frame onto its own input, and writes multi-MODEL Cα PDBs, an `aligned_coordinates.npz`, and a per-frame RMSD table. Everything below reads that one export, so the geometry is consistent across all seven metric plots.

This is the same suite as `CNN2d_AE_BRAF.ipynb`: RMSD, radius of gyration, per-residue RMSF, Cα bond lengths, Cα angles, Cα pseudo-dihedrals, and writhe chirality.

In [ ]:
aligned = wf.export_kabsch_aligned_datasets()
print("Export keys:", sorted(aligned.keys()))

### 3.1 Per-frame RMSD <a id="31-per-frame-rmsd"></a>

Distance between each input frame and its Kabsch-aligned reconstruction. This is the headline accuracy number for the model.

In [ ]:
wf.plot_rmsd_comparison(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

### 3.2 Radius of gyration <a id="32-radius-of-gyration"></a>

Global compactness. A decoder that collapses or inflates structures shows up here even when the per-frame RMSD looks acceptable.

In [ ]:
wf.plot_rg_comparison(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

### 3.3 Per-residue RMSF <a id="33-per-residue-rmsf"></a>

Where along the loop the ensemble varies, input versus decoded. Systematic flattening means the latent space is not carrying the flexible regions.

In [ ]:
wf.plot_rmsf_comparison(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

### 3.4 Cα bond lengths <a id="34-ca-bond-lengths"></a>

Consecutive Cα–Cα distances. Real protein backbones sit tightly around 3.8 Å; a broad or shifted decoded distribution means the reconstructions are not physically plausible chains.

In [ ]:
wf.plot_ca_bondlength_comparison(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

### 3.5 Cα angles <a id="35-ca-angles"></a>

Cα(i)–Cα(i+1)–Cα(i+2) valence angles: the next-order local geometry check after bond lengths.

In [ ]:
wf.plot_ca_angle_comparison(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

### 3.6 Cα pseudo-dihedrals <a id="36-ca-pseudo-dihedrals"></a>

Cα(i)–Cα(i+1)–Cα(i+2)–Cα(i+3) torsions. These are the coarse-grained stand-in for backbone φ/ψ and are the most sensitive of the local descriptors: they carry the handedness of the chain, so a decoder that gets bond lengths and angles right can still fail here.

In [ ]:
wf.plot_ca_pseudodihedral_comparison(
    aligned_export=aligned, subfolder=SUBFOLDER, show=True
);

### 3.7 Writhe chirality <a id="37-writhe-chirality"></a>

The Klenin–Langowski writhe matrix split into its positive and negative parts and summed per frame. Writhe is invariant under rigid motion and under the global `(x − mean) / std` scaling, so it compares input and decoded ensembles without any alignment ambiguity, and the positive/negative split is a direct readout of whether the decoder preserves chirality.

In [ ]:
wf.plot_writhe_chirality(aligned_export=aligned, subfolder=SUBFOLDER, show=True);

## 4. The 6D latent space <a id="4-the-6d-latent-space"></a>

At `latent_dim=2` the latent space is a single scatter plot. At 6 it is not, so this section encodes both splits, keeps all six coordinates, and then looks at the space two dimensions at a time.

### 4.1 Encode the train and validation splits <a id="41-encode-the-train-and-validation-splits"></a>

`encode_datasets` runs the encoder over both splits and writes `latent_encoded_train.csv` / `latent_encoded_valid.csv` with **all six** columns (`z0 … z5`), not just the first two.

In [ ]:
enc = wf.encode_datasets()
z_train, z_valid = enc["z_train"], enc["z_valid"]

import numpy as np
import pandas as pd

display(
    pd.DataFrame(
        {
            "min": z_train.min(axis=0),
            "median": np.median(z_train, axis=0),
            "max": z_train.max(axis=0),
            "std": z_train.std(axis=0),
        },
        index=[f"z{i}" for i in range(z_train.shape[1])],
    ).round(4)
)

### 4.2 Reconstruction-error violins <a id="42-reconstruction-error-violins"></a>

Per-frame reconstruction RMSD for training versus validation, from the CSV written in section 3. Comparable distributions mean the 6D bottleneck generalises; a validation tail well above the training one is the signature of memorisation.

In [ ]:
wf.plot_error_violins(show=True);

### 4.3 Pairwise RMSD landscapes <a id="43-pairwise-rmsd-landscapes"></a>

One panel per dimension pair, 15 in total. For each pair the scan sweeps a `GRID_SIZE × GRID_SIZE` grid over that pair's encoded range (plus 10% padding) while **holding the other four dimensions at the median of the encoded training latents**, so every panel is a median slice through the 6D space.

The colour is molearn's `scan_error`: decode the grid point, re-encode the result, decode again, and measure the Cα RMSD between the two decodes, in Å. Low values mark regions the model maps consistently onto itself — the part of latent space where interpolated structures are trustworthy. High values mark regions the encoder and decoder disagree about, which is where sampling produces artefacts.

Training (blue) and validation (red) points are overlaid on the same axes, so you can see directly whether the data sits inside the well-behaved basin.

On orientation: surfaces are stored as `surface[iy, ix]` for `(xvals[ix], yvals[iy])` and drawn with `pcolormesh(xvals, yvals, surface)`, matching molearn's own plotting. The earlier single-run notebooks drew the same array with `imshow(surface.T)` and matplotlib's default `origin="upper"`, which both transposes the grid and flips the y axis — the scatter overlay then lands on the wrong cells.

In [ ]:
scan = wf.scan_latent_pairs(
    z_train,
    z_valid=z_valid,
    grid_size=GRID_SIZE,
    padding=GRID_PADDING,
    reference="median",
)
print("z_ref (median of training latents):", np.round(scan["z_ref"], 4))

In [ ]:
wf.plot_latent_pair_panels(scan, z_train, z_valid=z_valid, ncols=5, show=True);

### 4.4 Close-up on one dimension pair <a id="44-close-up-on-one-dimension-pair"></a>

The same surface for dimensions 0 and 1 at full size, with training and validation on separate axes sharing one colour scale. Change `pair` to inspect any of the 15 combinations.

In [ ]:
wf.plot_latent_pair(
    scan, z_train, z_valid=z_valid, pair=(0, 1), split_panels=True, show=True
);

## 5. Summary <a id="5-summary"></a>

**What the panels show.** Each of the 15 landscapes is a 2-D median slice of the 6D latent space, coloured by how far a decode–encode–decode round trip moves the structure. Dark regions are self-consistent; bright regions are where the model breaks down. Because the four non-scanned dimensions are fixed at the training median, a point that is bright in one panel is not necessarily unreachable — it may simply be off-slice — so the panels are best read together rather than individually.

**What to look for.** Training and validation points falling in the same dark basin means the 6D bottleneck generalises. Data pushed against a bright rim means the encoded range extends past where the decoder is reliable, which caps how far you can interpolate or sample. Comparing panel to panel also shows which dimensions actually carry structure: a pair whose landscape is flat and featureless is a pair the model is barely using.

**Artefacts.** Everything is written under `Results/cnn2d_latent6/cnn2d_latent6/`:

| File | Contents |
|---|---|
| `checkpoint_*.ckpt` | best model weights |
| `xbb_foldingnet_checkpoints/log*.dat` | per-epoch training log |
| `training_history.png` | section 2 |
| `aligned_pdbs/` | input and Kabsch-aligned decoded multi-MODEL Cα PDBs, plus `aligned_coordinates.npz` |
| `rmsd_per_frame_train_valid.csv` | per-frame reconstruction RMSD |
| `*_input_vs_decoded.png`, `ca_pseudodihedral_per_index_*.csv`, `writhe_chirality_per_frame.csv` | section 3 metrics |
| `latent_encoded_train.csv`, `latent_encoded_valid.csv` | all six latent coordinates per frame |
| `latent_landscape_pairs.npz` | the 15 RMSD surfaces with their axis values and the median reference vector |
| `latent_pair_landscapes.png`, `latent_landscape_pair.png` | sections 4.3 and 4.4 |

`latent_landscape_pairs.npz` can be reloaded without re-running the model via `workflow.replacingPCA.latent_landscape.load_scan(path)`, so the figures can be restyled without a GPU.